# 1x4 MMI Parameter Sweep

Optimize MMI length and width for uniform power splitting using the Design plugin.

In [1]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
import tidy3d.web as web
import tidy3d.plugins.design as tdd

td.config.local_cache.enabled = True

try:
    RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
except NameError:
    RESULTS_DIR = None

## Fixed Parameters

In [2]:
# Materials
n_sin = 2.0
n_sio2 = 1.44
sin = td.Medium(permittivity=n_sin**2)
sio2 = td.Medium(permittivity=n_sio2**2)

# Wavelength
lda0 = 1.55
freq0 = td.C_0 / lda0
freqs = [freq0]
fwidth = freq0 / 10

# Fixed geometry
wg_width = 1.0
wg_thickness = 0.4
taper_width = 1.5
taper_length = 5.0
buffer_x = 2.0
buffer_y = 2.0
buffer_z = 1.5
wg_extension = 10.0

## Simulation Builder Function

The function arguments must match the parameter names in the DesignSpace.

In [3]:
def make_sim(L_MMI: float, W_MMI: float) -> td.Simulation:
    """Build MMI simulation for given length and width."""

    output_spacing = W_MMI / 4
    out_positions = [
        -3 * output_spacing / 2,
        -output_spacing / 2,
        output_spacing / 2,
        3 * output_spacing / 2,
    ]

    Ly = taper_length + L_MMI + taper_length + 2 * buffer_y
    structures = []

    # Input waveguide with taper
    input_wg = td.Structure(
        geometry=td.PolySlab(
            vertices=[
                (-wg_width / 2, -wg_extension),
                (-wg_width / 2, 0),
                (-taper_width / 2, taper_length),
                (taper_width / 2, taper_length),
                (wg_width / 2, 0),
                (wg_width / 2, -wg_extension),
            ],
            axis=2,
            slab_bounds=(-wg_thickness / 2, wg_thickness / 2),
        ),
        medium=sin,
    )
    structures.append(input_wg)

    # MMI region
    mmi_region = td.Structure(
        geometry=td.Box(
            center=(0, taper_length + L_MMI / 2, 0),
            size=(W_MMI, L_MMI, wg_thickness),
        ),
        medium=sin,
    )
    structures.append(mmi_region)

    # Output waveguides
    output_y_start = taper_length + L_MMI
    output_taper_end = output_y_start + taper_length
    output_wg_end = Ly + wg_extension

    for x_pos in out_positions:
        output_wg = td.Structure(
            geometry=td.PolySlab(
                vertices=[
                    (x_pos - taper_width / 2, output_y_start),
                    (x_pos - wg_width / 2, output_taper_end),
                    (x_pos - wg_width / 2, output_wg_end),
                    (x_pos + wg_width / 2, output_wg_end),
                    (x_pos + wg_width / 2, output_taper_end),
                    (x_pos + taper_width / 2, output_y_start),
                ],
                axis=2,
                slab_bounds=(-wg_thickness / 2, wg_thickness / 2),
            ),
            medium=sin,
        )
        structures.append(output_wg)

    # Domain
    Lx = W_MMI + 2 * buffer_x
    Lz = wg_thickness + 2 * buffer_z
    sim_size = (Lx, Ly, Lz)

    # Source
    mode_spec = td.ModeSpec(num_modes=1, target_neff=n_sin)
    mode_source = td.ModeSource(
        center=(0, buffer_y / 2, 0),
        size=(4 * wg_width, 0, Lz),
        source_time=td.GaussianPulse(freq0=freq0, fwidth=fwidth),
        direction="+",
        mode_spec=mode_spec,
        mode_index=0,
    )

    # Flux monitors (no symmetry, measure all 4)
    flux_monitors = []
    for i, x_pos in enumerate(out_positions):
        monitor = td.FluxMonitor(
            center=(x_pos, output_taper_end + buffer_y / 2, 0),
            size=(3 * wg_width, 0, Lz),
            freqs=freqs,
            name=f"flux_out{i+1}",
        )
        flux_monitors.append(monitor)

    sim = td.Simulation(
        center=(0, Ly / 2 - buffer_y, 0),
        size=sim_size,
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=lda0),
        structures=structures,
        sources=[mode_source],
        monitors=flux_monitors,
        run_time=td.RunTimeSpec(quality_factor=10),
        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()),
        medium=sio2,
    )

    return sim

## Post-Processing Function

Compute metrics from simulation results.

In [4]:
def analyze(sim_data: td.SimulationData) -> dict:
    """Analyze MMI results and return metrics."""
    T = [float(sim_data[f"flux_out{i+1}"].flux.values.item()) for i in range(4)]

    total = sum(T)
    imbalance = max(T) - min(T)
    excess_loss = -10 * np.log10(total) if total > 0 else 99

    return {
        "T1": T[0],
        "T2": T[1],
        "T3": T[2],
        "T4": T[3],
        "total": total,
        "imbalance": imbalance,
        "excess_loss_dB": excess_loss,
    }

## Define Parameter Sweep

In [5]:
if __name__ == "__main__":
    # Parameters to sweep
    param_L = tdd.ParameterFloat(name="L_MMI", span=(40, 50), num_points=5)
    param_W = tdd.ParameterFloat(name="W_MMI", span=(7, 9), num_points=5)

    # Grid search method
    method = tdd.MethodGrid()

    # Create design space
    design_space = tdd.DesignSpace(
        parameters=[param_L, param_W],
        method=method,
        task_name="mmi_sweep",
    )

    print(f"Running {param_L.num_points * param_W.num_points} simulations...")

    # Run sweep
    result = design_space.run(fn=make_sim, fn_post=analyze, verbose=True)
    df = result.to_dataframe()

    # Save results
    if RESULTS_DIR:
        df.to_csv(f"{RESULTS_DIR}/sweep_results.csv")

    print("\nSweep Results:")
    print(df.to_string())

Running 25 simulations...


15:01:45 EST Running 25 Simulations


Sweep Results:
    L_MMI  W_MMI        T1        T2        T3        T4     total  imbalance  excess_loss_dB
0    40.0    7.0  0.324170  0.312272  0.312273  0.324171  1.272886   0.011899       -1.047893
1    42.5    7.0  0.262494  0.369533  0.369534  0.262494  1.264055   0.107041       -1.017660
2    45.0    7.0  0.238697  0.323067  0.323067  0.238696  1.123527   0.084371       -0.505835
3    47.5    7.0  0.247241  0.361942  0.361942  0.247241  1.218366   0.114700       -0.857777
4    50.0    7.0  0.226592  0.424357  0.424357  0.226592  1.301898   0.197765       -1.145771
5    40.0    7.5  0.299532  0.286736  0.286737  0.299532  1.172537   0.012796       -0.691266
6    42.5    7.5  0.393009  0.193412  0.193413  0.393008  1.172842   0.199597       -0.692396
7    45.0    7.5  0.341178  0.216633  0.216634  0.341178  1.115623   0.124545       -0.475173
8    47.5    7.5  0.265278  0.293886  0.293887  0.265279  1.118330   0.028608       -0.485698
9    50.0    7.5  0.245314  0.286836  0.2868

    ## Visualize Results
    

In [6]:
    # Find optimal design (minimize imbalance while keeping loss low)
    df_filtered = df[df["excess_loss_dB"] < 1.0]  # Filter high-loss designs
    if len(df_filtered) > 0:
        best_idx = df_filtered["imbalance"].idxmin()
        best = df_filtered.loc[best_idx]
        print(f"\nOptimal Design:")
        print(f"  L_MMI = {best['L_MMI']:.2f} μm")
        print(f"  W_MMI = {best['W_MMI']:.2f} μm")
        print(f"  Imbalance = {best['imbalance']:.4f}")
        print(f"  Excess Loss = {best['excess_loss_dB']:.2f} dB")
        print(f"  Outputs: {best['T1']:.3f}, {best['T2']:.3f}, {best['T3']:.3f}, {best['T4']:.3f}")

    # Plot heatmaps
    L_vals = df["L_MMI"].unique()
    W_vals = df["W_MMI"].unique()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Imbalance heatmap
    imbalance_grid = df.pivot(index="L_MMI", columns="W_MMI", values="imbalance")
    im1 = axes[0].pcolormesh(W_vals, L_vals, imbalance_grid.values, cmap="viridis_r")
    axes[0].set_xlabel("W_MMI (μm)")
    axes[0].set_ylabel("L_MMI (μm)")
    axes[0].set_title("Power Imbalance (lower is better)")
    plt.colorbar(im1, ax=axes[0])

    # Excess loss heatmap
    loss_grid = df.pivot(index="L_MMI", columns="W_MMI", values="excess_loss_dB")
    im2 = axes[1].pcolormesh(W_vals, L_vals, loss_grid.values, cmap="viridis_r")
    axes[1].set_xlabel("W_MMI (μm)")
    axes[1].set_ylabel("L_MMI (μm)")
    axes[1].set_title("Excess Loss (dB)")
    plt.colorbar(im2, ax=axes[1])

    plt.tight_layout()
    if RESULTS_DIR:
        plt.savefig(f"{RESULTS_DIR}/03_sweep_results.png", dpi=150)
    plt.close()

    print(f"\nResults saved to {RESULTS_DIR}/")


Optimal Design:
  L_MMI = 40.00 μm
  W_MMI = 9.00 μm
  Imbalance = 0.0009
  Excess Loss = -0.02 dB
  Outputs: 0.252, 0.251, 0.251, 0.252

Results saved to None/
